### This is where we chain every step

In [4]:
import sys
from pathlib import Path
import yaml
import pandas as pd
project_dir = Path.cwd().parent
sys.path.append(str(project_dir))
config_path = project_dir / "config.yaml"

from src.corrupt import random_missing_values
from src.clean import mean_fill
from src.evaluate import evaluate_continuous_recovery
from src.load import load_model_data
from src.run_experiment import run_experiment

with config_path.open() as config_file:
    config = yaml.safe_load(config_file)

corruption_rate = config["corruption_rate"]
target_drug = config["target_drug"]
columns = config["columns"]
seed = config["seed"]

processed_dir = project_dir / "data/processed/"
drug_consumption = project_dir / f"data/processed/Drug_Consumption_{target_drug}.csv"

# load the dfs
original_df = load_model_data(drug_consumption)

# corrupt the df
corrupted_df, mask_df = random_missing_values(original_df, corruption_rate, seed)

# clean the df 
cleaned_df = mean_fill(corrupted_df, columns, {})

# evaluate the cleaned df
evaluated_metrics = evaluate_continuous_recovery(original_df, cleaned_df, mask_df, columns)

# print("Continuous Recovery Metrics")
# print("---------------------------")
# print(f"RMSE:             {evaluated_metrics['rmse']:.4f}")
# print(f"MAE:              {evaluated_metrics['mae']:.4f}")
# print(f"Bias:             {evaluated_metrics['bias']:.4f}")
# print(f"Variance Ratio:   {evaluated_metrics['variance_ratio']:.4f}")
# print(f"Wasserstein:      {evaluated_metrics['wasserstein']:.4f}")

result_df = run_experiment(original_df)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

parquet = pd.read_parquet(project_dir / "results/experiments.parquet")

parquet

,seed,corruption_rate,strategy,corruption_type,rmse,mae,bias,variance_ratio,wasserstein
0,1,0.1,mean_fill,random_missing_values,0.997247,0.803705,0.035238,0.000031,0.799213
1,1,0.2,mean_fill,random_missing_values,0.988324,0.794781,0.019566,0.000126,0.784842
2,1,0.3,mean_fill,random_missing_values,0.985582,0.790442,0.008101,0.000089,0.782429
3,2,0.1,mean_fill,random_missing_values,0.978128,0.788099,0.007191,0.000054,0.782146
4,2,0.2,mean_fill,random_missing_values,0.980858,0.786616,0.021737,0.000127,0.780568
5,2,0.3,mean_fill,random_missing_values,0.989572,0.794157,0.018551,0.000034,0.790086
6,3,0.1,mean_fill,random_missing_values,0.978401,0.788798,0.022872,0.000137,0.779081
7,3,0.2,mean_fill,random_missing_values,0.969645,0.782452,0.038442,0.000097,0.776285
8,3,0.3,mean_fill,random_missing_values,0.984645,0.793580,0.026359,0.000233,0.782513
9,4,0.1,mean_fill,random_missing_values,0.956391,0.766359,-0.026265,0.000061,0.760894
